# 서울시 따릉이 스테이션별 대여 수요 예측 파이프라인

## 공통 베이스라인(데이터 전처리 및 기본 학습)

### 환경 설정 및 라이브러리 임포트

In [1]:
# ==========================================
# 통합 라이브러리 설정
# ==========================================
import os
import sys
import re
import time
import gc
import requests
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from datetime import datetime, date, timedelta
from concurrent.futures import ThreadPoolExecutor
import geopandas as gpd
from shapely import wkt
from IPython.display import display, HTML

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from functools import reduce

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
from tqdm.notebook import tqdm

# 프로젝트 경로 설정 및 환경변수 로드
sys.path.append(os.path.dirname(os.getcwd()))
load_dotenv()

# 시각화 및 전역 환경 설정
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams['font.family'] = 'Malgun Gothic'

SEED = 42
np.random.seed(SEED)

print("\n========== 데이터 분석 환경 설정 완료 ==========")


========== 데이터 분석 환경 설정 완료 ==========


### DB 연결 및 원본 데이터 로드

In [2]:
# ==========================================
# 환경 설정 및 DB 연결
# ==========================================
DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")
DB_HOST     = os.getenv("DB_HOST", "localhost")
DB_PORT     = os.getenv("DB_PORT", "3306")
DB_NAME     = os.getenv("DB_NAME", "seoul_bike")

DATABASE_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    if conn.execute(text("SELECT 1")).scalar() == 1:
        print("\n========== 데이터베이스 연결 성공 ==========")
    else:
        print("\n========== 데이터베이스 연결 실패 ==========")

print("========== 각 테이블 데이터 로드 시작 ==========")

# 환경, 인프라, 인구 데이터
hourly_air_2024_df    = pd.read_sql_table("hourly_air_2024", con=engine)
hourly_precip_2024_df = pd.read_sql_table("hourly_precip_2024", con=engine)
hourly_snow_2024_df   = pd.read_sql_table("hourly_snow_2024", con=engine)
hourly_temp_2024_df   = pd.read_sql_table("hourly_temp_2024", con=engine)
rt_air_df             = pd.read_sql_table("rt_air", con=engine)
rt_weather_df         = pd.read_sql_table("rt_weather", con=engine)
infra_business_df     = pd.read_sql_table("infra_business", con=engine)
infra_park_df         = pd.read_sql_table("infra_park", con=engine)
infra_river_df        = pd.read_sql_table("infra_river", con=engine)
infra_school_df       = pd.read_sql_table("infra_school", con=engine)
infra_univ_df         = pd.read_sql_table("infra_univ", con=engine)
infra_subway_df       = pd.read_sql_table("infra_subway", con=engine)
pop_flow_2024_df      = pd.read_sql_table("pop_flow_2024", con=engine)
pop_living_2024_df    = pd.read_sql_table("pop_living_2024", con=engine)
korea_holidays_df     = pd.read_sql_table("korea_holidays", con=engine)
station_loc_df        = pd.read_sql_table("station_loc", con=engine)

# 따릉이 이력 데이터
table_name = "rent_history_2024"
chunk_size = 100000
start_time = time.time()

chunk_iterator = pd.read_sql_table(table_name, con=engine, chunksize=chunk_size)
df_list = [chunk for chunk in chunk_iterator]
rent_history_2024_df = pd.concat(df_list, ignore_index=True)

print("\n========== 로드된 데이터프레임 요약 ==========")
df_dict = {
    "hourly_air_2024": hourly_air_2024_df,
    "hourly_precip_2024": hourly_precip_2024_df,
    "hourly_snow_2024": hourly_snow_2024_df,
    "hourly_temp_2024": hourly_temp_2024_df,
    "rt_air": rt_air_df,
    "rt_weather": rt_weather_df,
    "infra_business": infra_business_df,
    "infra_park": infra_park_df,
    "infra_river": infra_river_df,
    "infra_school": infra_school_df,
    "infra_univ": infra_univ_df,
    "infra_subway": infra_subway_df,
    "pop_flow_2024": pop_flow_2024_df,
    "pop_living_2024": pop_living_2024_df,
    "korea_holidays": korea_holidays_df,
    "station_loc": station_loc_df,
    "rent_history_2024": rent_history_2024_df
}

summary_data = [
    {"Table Name": name, "Row Count": len(df)}
    for name, df in df_dict.items()
]
summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values(by="Row Count", ascending=False).reset_index(drop=True)
summary_df["Row Count"] = summary_df["Row Count"].apply(lambda x: f"{x:,}")
display(summary_df)


========== 데이터베이스 연결 성공 ==========
========== 각 테이블 데이터 로드 시작 ==========

========== [rent_history_2024] 테이블 데이터 로드 시작 ==========
총 로드된 행 수: 2,259,910행 / 소요 시간: 67.71초
========== 데이터 로드 완료 ==========


### 따릉이 대여소 위치 / 환경 / 인프라 데이터 전처리

In [3]:
# ==========================================
# 따릉이 대여소 위치 / 환경 / 인프라 데이터
# ==========================================
KAKAO_REST_API_KEY = os.getenv("KAKAO_REST_API_KEY")

# 따릉이 대여소 위치
print("\n========== 데이터 전처리 및 병합 시작 ==========")
HARDCODED_COORDS = {
    'ST-1066': (37.55290, 126.83650), 'ST-1068': (37.54897, 126.84852),
    'ST-1073': (37.50325, 127.12782), 'ST-1074': (37.49830, 127.13454),
    'ST-1090': (37.48083, 127.12933), 'ST-1091': (37.50743, 127.10123),
    'ST-1255': (37.56847, 126.84803), 'ST-1318': (37.53424, 126.89736),
    'ST-1412': (37.50383, 127.13876), 'ST-1415': (37.48161, 127.14361),
    'ST-2':    (37.55088, 126.91039), 'ST-415':  (37.51980, 126.88937),
    'ST-423':  (37.52784, 126.92873), 'ST-989':  (37.54955, 126.91071)
}
station_loc_df['station_id'] = station_loc_df['station_id'].astype(str).str.strip()
missing_mask = (station_loc_df['lat'] == 0.0) | (station_loc_df['lon'] == 0.0)

for st_id, coords in HARDCODED_COORDS.items():
    idx = station_loc_df[station_loc_df['station_id'] == st_id].index
    if not idx.empty:
        station_loc_df.loc[idx, 'lat'] = coords[0]
        station_loc_df.loc[idx, 'lon'] = coords[1]

# 환경 데이터 전처리
def preprocess_env(df, col_name):
    df = df.copy()
    if 'id' in df.columns: df = df.drop(columns=['id'])
    df['measure_date'] = pd.to_datetime(df['measure_date'])
    df.replace([-9, -9.0], np.nan, inplace=True)
    return df.set_index('measure_date').groupby('region_name')[col_name].resample('1h').mean().reset_index()

env_dfs = [
    preprocess_env(hourly_air_2024_df, 'pm10'),
    preprocess_env(hourly_temp_2024_df, 'temperature'),
    preprocess_env(hourly_precip_2024_df, 'precipitation'),
    preprocess_env(hourly_snow_2024_df, 'snowfall')
]
env_master_2024_df = reduce(lambda l, r: pd.merge(l, r, on=['measure_date', 'region_name'], how='outer'), env_dfs)
env_master_2024_df = env_master_2024_df.sort_values(by=['region_name', 'measure_date']).reset_index(drop=True)
env_master_2024_df['temperature'] = env_master_2024_df.groupby('region_name')['temperature'].transform(lambda x: x.interpolate(method='linear').ffill().bfill())
env_master_2024_df['pm10'] = env_master_2024_df.groupby('region_name')['pm10'].transform(lambda x: x.interpolate(method='linear').ffill().bfill())
env_master_2024_df['precipitation'] = env_master_2024_df['precipitation'].fillna(0)
env_master_2024_df['snowfall'] = env_master_2024_df['snowfall'].fillna(0)

# 인프라 공간 데이터 변환
def preprocess_gdf(df, wkt_col=None):
    if df.empty: return gpd.GeoDataFrame()
    if wkt_col:
        df = df.dropna(subset=[wkt_col]).copy()
        df['geometry'] = df[wkt_col].apply(lambda x: wkt.loads(str(x)) if pd.notna(x) and str(x) != 'None' else None)
        df = df.dropna(subset=['geometry'])
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
    else:
        lat_col = 'latitude' if 'latitude' in df.columns else 'lat' if 'lat' in df.columns else None
        lon_col = 'longitude' if 'longitude' in df.columns else 'lon' if 'lon' in df.columns else 'lot' if 'lot' in df.columns else None
        if not lat_col or not lon_col: return gpd.GeoDataFrame()
        df = df.dropna(subset=[lat_col, lon_col]).copy()
        gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon_col].astype(float), df[lat_col].astype(float)), crs="EPSG:4326")
    return gdf.to_crs(epsg=5179)[~gdf.is_empty]

def count_infra(tgt, src, r, col):
    if src is None or src.empty: return pd.Series(0, index=tgt.index, name=col)
    buf = tgt.copy()
    buf['geometry'] = buf.geometry.buffer(r)
    joined = gpd.sjoin(buf, src, how='left', predicate='intersects')
    return joined.groupby(joined.index)['index_right'].count().rename(col)

def cal_nearest_infra(tgt, src, col):
    if src is None or src.empty: return pd.DataFrame({col: [np.nan]*len(tgt)}, index=tgt.index)
    nearest = gpd.sjoin_nearest(tgt, src, distance_col=col)
    return nearest[~nearest.index.duplicated(keep='first')][[col]]

station_loc_gdf = preprocess_gdf(station_loc_df)
infra_park_df['lon'], infra_park_df['lat'] = infra_park_df['xcrd_g'].fillna(infra_park_df['xcrd']), infra_park_df['ycrd_g'].fillna(infra_park_df['ycrd'])
infra_subway_df['lon'] = pd.to_numeric(infra_subway_df.get('lon', infra_subway_df.get('lot')), errors='coerce')

infra_park_gdf = preprocess_gdf(infra_park_df)
infra_river_gdf = preprocess_gdf(infra_river_df, wkt_col='geom_wkt')
infra_subway_gdf = preprocess_gdf(infra_subway_df)
infra_business_gdf = preprocess_gdf(infra_business_df)
infra_edu_gdf = pd.concat([preprocess_gdf(infra_school_df), preprocess_gdf(infra_univ_df)], ignore_index=True)

infra_master_df = station_loc_gdf.copy()
infra_master_df['subway_cnt_300m'] = count_infra(infra_master_df, infra_subway_gdf, 300, 'subway_cnt_300m')
infra_master_df['biz_cnt_300m']    = count_infra(infra_master_df, infra_business_gdf, 300, 'biz_cnt_300m')
infra_master_df['edu_cnt_500m']    = count_infra(infra_master_df, infra_edu_gdf, 500, 'edu_cnt_500m')
infra_master_df['park_cnt_500m']   = count_infra(infra_master_df, infra_park_gdf, 500, 'park_cnt_500m')
infra_master_df['river_cnt_1km']   = count_infra(infra_master_df, infra_river_gdf, 1000, 'river_cnt_1km')
infra_master_df = infra_master_df.join(cal_nearest_infra(infra_master_df, infra_subway_gdf, 'dist_subway'))
infra_master_df = infra_master_df.join(cal_nearest_infra(infra_master_df, infra_river_gdf, 'dist_river'))
infra_master_df = infra_master_df.drop(columns=['geometry'])

print("========== 전처리 및 병합 완료 ==========")


========== 데이터 전처리 및 병합 시작 ==========
========== 전처리 및 병합 완료 ==========


### 인구 데이터 전처리

In [4]:
# ==========================================
# 인구 데이터
# ==========================================
print("\n========== 인구 데이터 전처리 및 병합 시작 ==========")

# 생활인구 정리
district_map = {'11500': '강서구', '11560': '영등포구', '11440': '마포구', '11710': '송파구'}
pop_living_2024_df['district_name'] = pop_living_2024_df['adstrd_code_se'].map(district_map)
pop_living_2024_df.rename(columns={'tot_lvpop_co': 'lvgpop_tot'}, inplace=True)
pop_living_2024_df['lvgpop_10s'] = pop_living_2024_df[['male_f10t14_lvpop_co', 'male_f15t19_lvpop_co', 'female_f10t14_lvpop_co', 'female_f15t19_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_20s'] = pop_living_2024_df[['male_f20t24_lvpop_co', 'male_f25t29_lvpop_co', 'female_f20t24_lvpop_co', 'female_f25t29_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_30s'] = pop_living_2024_df[['male_f30t34_lvpop_co', 'male_f35t39_lvpop_co', 'female_f30t34_lvpop_co', 'female_f35t39_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_40s'] = pop_living_2024_df[['male_f40t44_lvpop_co', 'male_f45t49_lvpop_co', 'female_f40t44_lvpop_co', 'female_f45t49_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_50s'] = pop_living_2024_df[['male_f50t54_lvpop_co', 'male_f55t59_lvpop_co', 'female_f50t54_lvpop_co', 'female_f55t59_lvpop_co']].sum(axis=1)
pop_living_2024_df['lvgpop_60up'] = pop_living_2024_df[['male_f60t64_lvpop_co', 'male_f65t69_lvpop_co', 'male_f70t74_lvpop_co', 'female_f60t64_lvpop_co', 'female_f65t69_lvpop_co', 'female_f70t74_lvpop_co']].sum(axis=1)

pop_living_2024_df = pop_living_2024_df[['stdr_de_id', 'tmzon_pd_se', 'district_name', 'lvgpop_tot', 'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up']].copy()
pop_living_2024_df['date_str'] = pop_living_2024_df['stdr_de_id'].astype(str)
pop_living_2024_df['hour_str'] = pop_living_2024_df['tmzon_pd_se'].astype(str).str.zfill(2)

# 마스터 뼈대 생성
date_rng = pd.date_range(start='2024-01-01 00:00:00', end='2024-12-31 23:00:00', freq='h')
districts = ['강서구', '영등포구', '마포구', '송파구']
pop_master_list = []
for dist in districts:
    df_temp = pd.DataFrame(date_rng, columns=['datetime'])
    df_temp['district_name'] = dist
    df_temp['date_str'] = df_temp['datetime'].dt.strftime('%Y%m%d')
    df_temp['hour_str'] = df_temp['datetime'].dt.strftime('%H')
    df_temp['weekday'] = df_temp['datetime'].dt.weekday
    df_temp['quarter_str'] = '2024' + df_temp['datetime'].dt.quarter.astype(str)
    pop_master_list.append(df_temp)
pop_master_2024_df = pd.concat(pop_master_list, ignore_index=True)

# 유동인구 병합 및 벡터화 분배 계산
pop_flow_2024_df = pd.merge(pop_master_2024_df, pop_flow_2024_df, left_on=['quarter_str', 'district_name'], right_on=['stdr_yyqu_cd', 'signgu_cd_nm'], how='left')

h = pop_flow_2024_df['hour_str'].astype(int)
wd = pop_flow_2024_df['weekday']

div = np.select([h.isin(range(6, 11)), h.isin(range(11, 17)), h.isin(range(17, 21)), h.isin(range(21, 24))], [5, 3, 4, 3], default=6)
time_pop = np.select(
    [h.isin(range(0, 6)), h.isin(range(6, 11)), h.isin(range(11, 14)), h.isin(range(14, 17)), h.isin(range(17, 21))],
    [pop_flow_2024_df['tmzon_00_06_flpop_co'], pop_flow_2024_df['tmzon_06_11_flpop_co'], pop_flow_2024_df['tmzon_11_14_flpop_co'], pop_flow_2024_df['tmzon_14_17_flpop_co'], pop_flow_2024_df['tmzon_17_21_flpop_co']],
    default=pop_flow_2024_df['tmzon_21_24_flpop_co']
)
day_pop = np.select(
    [wd == 0, wd == 1, wd == 2, wd == 3, wd == 4, wd == 5, wd == 6],
    [pop_flow_2024_df['mon_flpop_co'], pop_flow_2024_df['tues_flpop_co'], pop_flow_2024_df['wed_flpop_co'], pop_flow_2024_df['thur_flpop_co'], pop_flow_2024_df['fri_flpop_co'], pop_flow_2024_df['sat_flpop_co'], pop_flow_2024_df['sun_flpop_co']],
    default=0
)

tot_pop = pop_flow_2024_df['tot_flpop_co'].fillna(0)
valid = tot_pop > 0

# 벡터 연산으로 유동인구 분배
est_tot_flwpop = np.zeros(len(pop_flow_2024_df))
est_tot_flwpop[valid] = (time_pop[valid] / div[valid]) * (day_pop[valid] / tot_pop[valid]) / 13
pop_flow_2024_df['flwpop_tot'] = est_tot_flwpop

age_map = {'agrde_10_flpop_co': 'flwpop_10s', 'agrde_20_flpop_co': 'flwpop_20s', 'agrde_30_flpop_co': 'flwpop_30s', 'agrde_40_flpop_co': 'flwpop_40s', 'agrde_50_flpop_co': 'flwpop_50s', 'agrde_60_above_flpop_co': 'flwpop_60up'}
for origin, new in age_map.items():
    pop_flow_2024_df[new] = np.where(valid, est_tot_flwpop * (pop_flow_2024_df[origin] / tot_pop), 0)

# 최종 인구 마스터 병합
pop_master_2024_df = pd.merge(pop_flow_2024_df, pop_living_2024_df, on=['date_str', 'hour_str', 'district_name'], how='left')
pop_cols = ['datetime', 'district_name', 'flwpop_tot', 'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up', 'lvgpop_tot', 'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up']
pop_master_2024_df = pop_master_2024_df[pop_cols]

print("========== 인구 데이터 전처리 및 병합 완료 ==========")


========== 인구 데이터 전처리 및 병합 시작 ==========
========== 인구 데이터 전처리 및 병합 완료 ==========


### 수요 예측용 마스터 데이터 통합

In [5]:
# ==========================================
# 수요 예측용 마스터 데이터 병합
# ==========================================
print("\n========== 수요 예측용 마스터 데이터 병합 시작 ==========")

rent_history_2024_df['station_id'] = rent_history_2024_df['station_id'].astype(str).str.strip()

# 병합
demand_predict_master_2024_df = pd.merge(rent_history_2024_df, station_loc_df[['station_id', 'district', 'lat', 'lon']], on='station_id', how='left')

infra_cols = ['station_id', 'subway_cnt_300m', 'biz_cnt_300m', 'edu_cnt_500m', 'park_cnt_500m', 'river_cnt_1km', 'dist_subway', 'dist_river']
demand_predict_master_2024_df = pd.merge(demand_predict_master_2024_df, infra_master_df[infra_cols], on='station_id', how='left')

demand_predict_master_2024_df['datetime_hr'] = pd.to_datetime(demand_predict_master_2024_df['datetime_hr'])
demand_predict_master_2024_df = pd.merge(demand_predict_master_2024_df, env_master_2024_df[['measure_date', 'region_name', 'temperature', 'precipitation', 'snowfall', 'pm10']], left_on=['datetime_hr', 'district'], right_on=['measure_date', 'region_name'], how='left')
demand_predict_master_2024_df.drop(columns=['measure_date', 'region_name'], inplace=True)

demand_predict_master_2024_df = pd.merge(demand_predict_master_2024_df, pop_master_2024_df, left_on=['datetime_hr', 'district'], right_on=['datetime', 'district_name'], how='left')
demand_predict_master_2024_df.drop(columns=['datetime', 'district_name'], inplace=True)

demand_predict_master_2024_df['temp_date'] = demand_predict_master_2024_df['datetime_hr'].dt.date
korea_holidays_df['holiday_date'] = pd.to_datetime(korea_holidays_df['holiday_date']).dt.date
demand_predict_master_2024_df = pd.merge(demand_predict_master_2024_df, korea_holidays_df[['holiday_date', 'holiday_name']], left_on='temp_date', right_on='holiday_date', how='left')

# 파생 변수 추가
demand_predict_master_2024_df['is_holiday'] = demand_predict_master_2024_df['holiday_name'].notna().astype(int)
demand_predict_master_2024_df['day_of_week'] = demand_predict_master_2024_df['datetime_hr'].dt.dayofweek
demand_predict_master_2024_df['is_weekend'] = (demand_predict_master_2024_df['day_of_week'] >= 5).astype(int)
demand_predict_master_2024_df['month'] = demand_predict_master_2024_df['datetime_hr'].dt.month
demand_predict_master_2024_df['hour'] = demand_predict_master_2024_df['datetime_hr'].dt.hour
demand_predict_master_2024_df.drop(columns=['temp_date', 'holiday_date', 'holiday_name', 'district'], inplace=True)

# 결측치 채우기
fill_zero_cols = ['precipitation', 'snowfall', 'subway_cnt_300m', 'biz_cnt_300m', 'edu_cnt_500m', 'park_cnt_500m', 'river_cnt_1km']
demand_predict_master_2024_df[fill_zero_cols] = demand_predict_master_2024_df[fill_zero_cols].fillna(0)

# 메모리 정리
del rent_history_2024_df, env_master_2024_df, pop_master_2024_df, infra_master_df
gc.collect()

print("========== 마스터 데이터 병합 및 메모리 정리 완료 ==========")


========== 수요 예측용 마스터 데이터 병합 시작 ==========
========== 마스터 데이터 병합 및 메모리 정리 완료 ==========


### 시계열 기반 데이터 분할 (Split)

In [6]:
# ==========================================
# 머신러닝 피처 세팅 및 분할 (Train/Val/Test)
# ==========================================
print("\n========== 시계열 데이터 정렬 및 분할 시작 ==========")

FEATURE_COLUMNS = [
    'lat', 'lon', 'subway_cnt_300m', 'biz_cnt_300m', 'edu_cnt_500m', 'park_cnt_500m', 'river_cnt_1km',
    'dist_subway', 'dist_river', 'temperature', 'precipitation', 'snowfall', 'pm10',
    'flwpop_10s', 'flwpop_20s', 'flwpop_30s', 'flwpop_40s', 'flwpop_50s', 'flwpop_60up',
    'lvgpop_10s', 'lvgpop_20s', 'lvgpop_30s', 'lvgpop_40s', 'lvgpop_50s', 'lvgpop_60up',
    'is_holiday', 'day_of_week', 'is_weekend', 'month', 'hour'
]
TARGET_COLUMNS = ['general_rent_cnt', 'sprout_rent_cnt', 'general_rtn_cnt', 'sprout_rtn_cnt']

# 시계열 정렬
df_sorted = demand_predict_master_2024_df.sort_values("datetime_hr").reset_index(drop=True)

X = df_sorted[FEATURE_COLUMNS]
Y = df_sorted[TARGET_COLUMNS]

# 컬럼명 문자열 변환
X.columns = [str(col) for col in X.columns]

n = len(X)
train_end = int(n * 0.60)
val_end   = int(n * 0.80)

X_train = X.iloc[:train_end]
Y_train = Y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
Y_val = Y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
Y_test = Y.iloc[val_end:]

print(f"Train 데이터: {len(X_train):,}행 ({df_sorted['datetime_hr'].iloc[0]} ~ {df_sorted['datetime_hr'].iloc[train_end-1]})")
print(f"Val   데이터: {len(X_val):,}행 ({df_sorted['datetime_hr'].iloc[train_end]} ~ {df_sorted['datetime_hr'].iloc[val_end-1]})")
print(f"Test  데이터: {len(X_test):,}행 ({df_sorted['datetime_hr'].iloc[val_end]} ~ {df_sorted['datetime_hr'].iloc[-1]})")

# 원본 데이터프레임 메모리 해제
del df_sorted, demand_predict_master_2024_df
gc.collect()


========== 시계열 데이터 정렬 및 분할 시작 ==========
Train 데이터: 1,355,946행 (2024-01-01 00:00:00 ~ 2024-07-28 18:00:00)
Val   데이터: 451,982행 (2024-07-28 18:00:00 ~ 2024-10-10 12:00:00)
Test  데이터: 451,982행 (2024-10-10 12:00:00 ~ 2025-01-01 15:00:00)


0

### 베이스라인 모델 학습 및 MLflow 실험 트래킹

In [7]:
# ==========================================
# MLflow 세팅, 평가 함수 및 최종 학습 루프
# ==========================================
USE_TEAM_SERVER = True
TEAM_SERVER_URI = "http://223.194.48.21:5000"
AUTHOR = "장수연"

if USE_TEAM_SERVER:
    mlflow.set_tracking_uri(TEAM_SERVER_URI)
else:
    mlflow.set_tracking_uri("sqlite:///mlflow_seoul_bike.db")

mlflow.set_experiment("bike_demand_prediction")

def rmsle(y, pred):
    log_y = np.log1p(y)
    log_pred = np.log1p(np.maximum(pred, 0))
    return np.sqrt(np.mean((log_y - log_pred) ** 2))

def evaluate_regr(y, pred):
    return {
        "rmsle": rmsle(y, pred),
        "rmse": np.sqrt(mean_squared_error(y, pred)),
        "mae": mean_absolute_error(y, pred)
    }

# 최적화된 모델 구성
models = {
    "LinearRegression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=0.1)),
    "RandomForest": RandomForestRegressor(n_estimators=50, max_depth=10, random_state=SEED, n_jobs=-1), # 파라미터 축소로 속도 개선
    "LightGBM": LGBMRegressor(n_estimators=100, random_state=SEED, n_jobs=-1, verbosity=-1),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=SEED, n_jobs=-1, tree_method='hist')
}

print("\n========== 단일 통합 타깃별 모델 학습 및 MLflow 기록 시작 ==========")
results = []
run_date = datetime.now().strftime("%Y-%m-%d %H:%M")

for target_name in TARGET_COLUMNS:
    print(f"\n========== [{target_name}] 예측 학습 시작 ==========")
    y_train_target = Y_train[target_name]
    y_val_target = Y_val[target_name]

    for model_name, model in models.items():
        run_name = f"{model_name}_{target_name}"
        try:
            with mlflow.start_run(run_name=run_name):
                print(f" >>> {model_name} 학습 중")
                model.fit(X_train, y_train_target)
                pred = model.predict(X_val)
                metrics = evaluate_regr(y_val_target, pred)

                # MLflow 기록
                mlflow.log_param("target", target_name)
                mlflow.log_param("model", model_name)
                mlflow.set_tag("author", AUTHOR)
                mlflow.set_tag("run_date", run_date)
                mlflow.log_metric("rmsle", metrics["rmsle"])
                mlflow.log_metric("rmse",  metrics["rmse"])
                mlflow.log_metric("mae",   metrics["mae"])

                results.append({
                    "target": target_name,
                    "model_name": model_name,
                    "rmsle": metrics["rmsle"],
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"]
                })
        except Exception as e:
            print(f" >>> [{model_name}/{target_name}] 학습 실패: {e}")

print("\n========== 타깃 모델 학습 및 MLflow 기록 종료 ==========")

# 결과 출력
results_df = pd.DataFrame(results)
display(results_df.sort_values(by=['target', 'rmsle']))


========== 단일 통합 타깃별 모델 학습 및 MLflow 기록 시작 ==========

========== [general_rent_cnt] 예측 학습 시작 ==========
 >>> LinearRegression 학습 중
🏃 View run LinearRegression_general_rent_cnt at: http://223.194.48.21:5000/#/experiments/1/runs/40415016748949ecae7f256d2354e145
🧪 View experiment at: http://223.194.48.21:5000/#/experiments/1
 >>> Ridge 학습 중
🏃 View run Ridge_general_rent_cnt at: http://223.194.48.21:5000/#/experiments/1/runs/1e600ce5d3e34219a7c32ec8bb338fa2
🧪 View experiment at: http://223.194.48.21:5000/#/experiments/1
 >>> Lasso 학습 중
🏃 View run Lasso_general_rent_cnt at: http://223.194.48.21:5000/#/experiments/1/runs/f7b5bf933b0c449caf8baa8470bc74cf
🧪 View experiment at: http://223.194.48.21:5000/#/experiments/1
 >>> RandomForest 학습 중
🏃 View run RandomForest_general_rent_cnt at: http://223.194.48.21:5000/#/experiments/1/runs/b45727f4082c40988f9eddbd9779da89
🧪 View experiment at: http://223.194.48.21:5000/#/experiments/1
 >>> LightGBM 학습 중
🏃 View run LightGBM_general_rent_cnt at: http://

,target,model_name,rmsle,rmse,mae
5,general_rent_cnt,XGBoost,0.644482,3.716881,2.161704
4,general_rent_cnt,LightGBM,0.650723,3.777135,2.216034
3,general_rent_cnt,RandomForest,0.726749,4.303685,2.571392
0,general_rent_cnt,LinearRegression,0.835022,5.214986,3.004279
1,general_rent_cnt,Ridge,0.835033,5.214972,3.004323
2,general_rent_cnt,Lasso,0.873932,5.287793,3.152117
17,general_rtn_cnt,XGBoost,0.611156,3.552522,2.048344
16,general_rtn_cnt,LightGBM,0.637606,3.683887,2.161921
15,general_rtn_cnt,RandomForest,0.712987,4.300676,2.516116
12,general_rtn_cnt,LinearRegression,0.827594,5.203675,2.993731


## [팀원 개별 작업] 모델 튜닝 및 성능 개선

### 하이퍼파라미터 튜닝

### 앙상블

### 최종 모델 선택

### Test셋 최종 평가

### 모델 저장(pkl) 및 검증